# Mini-CLAP

Today, we will be implementing our own version of CLAP (Contrastive Language-Audio Pretraining). The original CLAP model is a multimodal model that learns to align speech and text embeddings in a shared space. The model is trained with contrastive learning, where positive pairs are formed by aligning the speech and text embeddings. The model is trained with a contrastive loss that encourages the positive pairs to be closer to each other than the negative pairs.

## Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## Text Encoder

We will be using the 

In [3]:
from transformers import BertTokenizer, BertModel

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained("bert-base-uncased").to(device)

In [4]:
text = "Example sentence"
encoded_input = tokenizer(text, return_tensors='pt').to(device)
with torch.no_grad():
    output = model(**encoded_input)

In [5]:
# getting the CLS token
output.last_hidden_state[:, 0, :].shape

torch.Size([1, 768])

## Audio Encoder

In [6]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from datasets import load_dataset

# load model + processor
model_id = "openai/whisper-small"
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id).to(device)
encoder = model.get_encoder()

processor = AutoProcessor.from_pretrained(model_id)

# load dataset
dataset = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
sample = dataset[0]["audio"]["array"]

# preprocess inputs
input_features = processor(sample, return_tensors="pt").input_features
input_features = input_features.to(device)

# forward pass to get encoder hidden states
with torch.no_grad():
    encoder_hidden_states = encoder(input_features).last_hidden_state

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


In [10]:
encoder_hidden_states.mean(axis=1).shape

torch.Size([1, 768])

## Building the Model

In [11]:
class ProjectionNN(nn.Module):
    def __init__(self):
        super(ProjectionNN, self).__init__()
        self.fc1a = nn.Linear(768, 256)
        self.fc1b = nn.Linear(256, 100)

        self.fc2a = nn.Linear(768, 256)
        self.fc2b = nn.Linear(256, 100)

    def forward(self, x1, x2):
        x1 = F.relu(self.fc1a(x1))
        x1 = F.relu(self.fc1b(x1))

        x2 = F.relu(self.fc2a(x2))
        x2 = F.relu(self.fc2b(x2))
        return x1, x2
    
class ContrastiveLoss(nn.Module):
    def __init__(self):
        super(ContrastiveLoss, self).__init__()
        self.cos = nn.CosineSimilarity()
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x1, x2):
        sim = self.cos(x1, x2)
        sim = self.softmax(sim)
        diag = torch.diag(sim)
        return torch.log(diag)
    

class miniCLAP(nn.Module):
    def __init__(self):
        super(miniCLAP, self).__init__()
        self.audio_encoder = AutoModelForSpeechSeq2Seq.from_pretrained(
            "openai/whisper-small")
        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.projection = ProjectionNN()

    def forward(self, audio, text):
        audio_encoded = encoder(audio).last_hidden_state.mean(axis=1)
        text_encoded = self.text_encoder(**text).last_hidden_state[:, 0, :]
        text_projection, audio_projection = self.projection(text_encoded, audio_encoded)
        scores = torch.matmul(text_projection.T, audio_projection)
        return scores

In [12]:
# train loop

model = miniCLAP().to(device)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Freeze parameters
for param in model.audio_encoder.parameters():
    param.requires_grad = False
for param in model.text_encoder.parameters():
    param.requires_grad = False

optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = ContrastiveLoss()

# testing forward pass

dataset = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
audio = dataset[0]["audio"]["array"]
audio = processor(sample, return_tensors="pt").input_features
audio = audio.to(device)

text = "Example sentence"
encoded_input = tokenizer(text, return_tensors='pt')
text = {key: val.to(device) for key, val in encoded_input.items()}

with torch.no_grad():
    output = model(audio, text)

It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


In [13]:
output

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        ...,
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 1.4161e-06,
         3.1721e-05],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 1.8574e-07,
         4.1608e-06]], device='cuda:0')